# Imports

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from olist.data import Olist

import utils.UVA as UVA
import utils.MVA as MVA
import utils.PCA as PCA

from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

In [4]:
data = Olist().get_data()

In [5]:
import warnings
warnings.filterwarnings('ignore')


%matplotlib inline
pd.options.mode.chained_assignment = None  # default='warn'

In [6]:
pd.set_option('display.max_rows', 750)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', None)

# Data Transformation

In [ ]:
columns_matching_table = [
    "order_id",
    "review_id",
    "customer_id",
    "product_id",
    "seller_id",
]

In [ ]:
#### Select only the columns of interest in the various dataframes of interest, before proceeding to any merge
orders = data['orders']
items = data['order_items']
reviews = data['order_reviews']
sellers = data['sellers']

#customers = data['customers'][['customer_id', 'customer_unique_id']]
customers = data['customers']
products = data['products']
product_names = data['product_category_name_translation']
payments = data['order_payments']
#geo = data['geolocation']

In [ ]:
data['orders']["order_purchase_timestamp"] = pd.to_datetime(data['orders']["order_purchase_timestamp"])
data['orders']["order_delivered_carrier_date"] = pd.to_datetime(data['orders']["order_delivered_carrier_date"])
data['orders']["order_delivered_customer_date"] = pd.to_datetime(data['orders']["order_delivered_customer_date"])
data['orders']["order_estimated_delivery_date"] = pd.to_datetime(data['orders']["order_estimated_delivery_date"])

In [ ]:
# Inspect the cardinality of each DataFrame using pd.DataFrame.shape and pd.Series.nunique()
print('orders:', orders.shape, orders.customer_id.nunique(), 'unique customer_ids, and', orders.order_id.nunique(), 'unique order_ids')
print('review: ', reviews.shape, reviews.order_id.nunique(), 'unique order_ids and', reviews.review_id.nunique(), 'unique reviews' )
print('items: ', items.shape, items.order_id.nunique(), 'unique order_ids,', items.product_id.nunique(), 
      'unique product_ids, and', items.seller_id.nunique(), 'unique seller_ids')

In [ ]:
# Carefully merge DataFrames
matching_table = orders.merge(reviews, on='order_id', how='left').merge(items, on='order_id', how='left')
matching_table

In [ ]:
matching_table = orders\
        .merge(customers, on='customer_id', how='outer')\
        .merge(reviews, on="order_id", how="outer")\
        .merge(items, on="order_id", how="outer")\
        .merge(sellers, on='seller_id', how='outer')
matching_table

In [ ]:
match_orders = orders.merge(customers, on='customer_id', how='left')

In [ ]:
orders_count = match_orders \
                .groupby('customer_unique_id')['order_id'] \
                .nunique().reset_index(name='orders_count')
orders_count['count_category'] = np.where(orders_count['orders_count'] == 1, '1', '2+')
orders_count.sort_values(by='orders_count', ascending=False)

In [ ]:
orders_info = pd.concat([orders_count['orders_count'].value_counts(normalize=True),
                        orders_count['orders_count'].value_counts()], 
                        axis=1,
                        keys=('percent','count customers'))
orders_info.index.name = 'orders per customer'
orders_info

In [ ]:
orders_info_cum = pd.concat([orders_count['count_category'].value_counts(normalize=True),
                        orders_count['count_category'].value_counts()], 
                        axis=1,
                        keys=('perc','count'))
orders_info_cum.index.name = 'orders per customer'
orders_info_cum

In [ ]:
products = products.merge(product_names, on='product_category_name', how='left')
products = products.reindex(['product_id', 'product_category_name',  'product_category_name_english', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'], axis=1)
products

In [ ]:
m_df = match_orders_2p \
    .merge(items[['order_id', 'product_id', 'price']], on='order_id', how='left') \
    .merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left') \
    .merge(reviews, on='order_id', how='left')
m_df

In [ ]:
product_count = m_df \
                .groupby('order_id')['product_id'] \
                .nunique().reset_index(name='prod_count')
product_count.sort_values(by='prod_count', ascending=False)

# Export